In [ ]:

import os
import json
import math
import time
import random
import argparse
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.transforms import functional as TF
from PIL import Image
from tqdm.auto import tqdm



# =============================================================================
DATA_ROOT = "/kaggle/input/datasets/jawadulkarim117/cityscapes/Cityscapes_Aug/Cityscapes_Aug"
OUTPUT_DIR = "/kaggle/working/outputs"

NUM_CLASSES      = 19
IGNORE_INDEX     = 255
IMAGE_SIZE       = (512, 1024)
BATCH_SIZE       = 4
NUM_WORKERS      = 4
EPOCHS           = 60
BASE_LR          = 1e-4
BACKBONE_LR_MULT = 0.3          # backbone unfrozen layers use BASE_LR * this
WEIGHT_DECAY     = 1e-4
WARMUP_EPOCHS    = 5
MIXED_PRECISION  = True
AUX_LOSS_WEIGHT  = 0.4
EDGE_LOSS_WEIGHT = 0.2
CRF_ITERATIONS   = 5
SEED             = 42
VAL_EVERY        = 1
LOG_EVERY        = 20

#  ume & Early Stopping ---   
AUTO_RESUME      = True          
EARLY_STOP_PATIENCE = 5        
# =============================================================================



LABELID_TO_TRAINID = {
    7: 0,  8: 1,  11: 2,  12: 3,  13: 4,
    17: 5, 19: 6, 20: 7,  21: 8,  22: 9,
    23: 10, 24: 11, 25: 12, 26: 13, 27: 14,
    28: 15, 31: 16, 32: 17, 33: 18,
}
CLASS_NAMES = [
    "road", "sidewalk", "building", "wall", "fence",
    "pole", "traffic light", "traffic sign", "vegetation", "terrain",
    "sky", "person", "rider", "car", "truck",
    "bus", "train", "motorcycle", "bicycle",
]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


class CityscapesFlat(Dataset):
    """Flat Cityscapes layout: <root>/<split>/images/*.png, <root>/<split>/masks/*.png."""

    def __init__(self, root, split="train", image_size=IMAGE_SIZE, is_train=True):
        self.img_dir = Path(root) / split / "images"
        self.mask_dir = Path(root) / split / "masks"
        assert self.img_dir.is_dir(), f"Missing {self.img_dir}"
        assert self.mask_dir.is_dir(), f"Missing {self.mask_dir}"

        self.images = sorted(self.img_dir.glob("*.png"))
        assert len(self.images) > 0, f"No images found in {self.img_dir}"

        self.image_size = image_size
        self.is_train = is_train

        self.label_lut = np.full(256, IGNORE_INDEX, dtype=np.uint8)
        for k, v in LABELID_TO_TRAINID.items():
            self.label_lut[k] = v

    def __len__(self):
        return len(self.images)

    def _mask_path(self, img_path: Path) -> Path:
        same_name = self.mask_dir / img_path.name
        if same_name.is_file():
            return same_name
        stem = img_path.name.replace("_leftImg8bit.png", "")
        return self.mask_dir / f"{stem}_gtFine_labelIds.png"

    def __getitem__(self, idx):
        img_path = self.images[idx]
        mask_path = self._mask_path(img_path)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        H, W = self.image_size
        image = image.resize((W, H), Image.BILINEAR)
        mask = mask.resize((W, H), Image.NEAREST)

        image = np.array(image, dtype=np.uint8)
        mask = np.array(mask, dtype=np.uint8)
        mask = self.label_lut[mask]

        if self.is_train:
            if random.random() < 0.5:
                image = np.ascontiguousarray(image[:, ::-1])
                mask = np.ascontiguousarray(mask[:, ::-1])
            image = self._color_jitter(image)

        image_t = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        image_t = TF.normalize(image_t, IMAGENET_MEAN, IMAGENET_STD)
        mask_t = torch.from_numpy(mask.astype(np.int64))

        boundary_t = self._mask_to_boundary(mask)
        return image_t, mask_t, boundary_t

    @staticmethod
    def _color_jitter(img_u8):
        img = img_u8.astype(np.float32)
        img *= random.uniform(0.85, 1.15)
        mean = img.mean(axis=(0, 1), keepdims=True)
        img = (img - mean) * random.uniform(0.85, 1.15) + mean
        return np.clip(img, 0, 255).astype(np.uint8)

    @staticmethod
    def _mask_to_boundary(mask):
        valid = (mask != IGNORE_INDEX)
        diff_h = np.zeros_like(mask, dtype=np.uint8)
        diff_w = np.zeros_like(mask, dtype=np.uint8)
        diff_h[:-1, :] = ((mask[:-1, :] != mask[1:, :]) & valid[:-1, :] & valid[1:, :]).astype(np.uint8)
        diff_w[:, :-1] = ((mask[:, :-1] != mask[:, 1:]) & valid[:, :-1] & valid[:, 1:]).astype(np.uint8)
        return torch.from_numpy((diff_h | diff_w).astype(np.float32))


# PATH A: ResNet-50 backbone (layer3, layer4 UNFROZEN for fine-tuning)

class ResNet50Backbone(nn.Module):
    """ResNet-50 with stem/layer1/layer2 frozen, layer3/layer4 trainable."""

    def __init__(self):
        super().__init__()
        from torchvision.models import resnet50, ResNet50_Weights
        m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool)
        self.layer1 = m.layer1
        self.layer2 = m.layer2
        self.layer3 = m.layer3
        self.layer4 = m.layer4

        # Freeze stem, layer1, layer2
        for module in [self.stem, self.layer1, self.layer2]:
            for p in module.parameters():
                p.requires_grad = False

        # layer3, layer4 remain trainable (requires_grad=True by default)

    def train(self, mode=True):
        """Keep frozen parts in eval (BN stats fixed), unfrozen parts follow mode."""
        super().train(mode)
        self.stem.eval()
        self.layer1.eval()
        self.layer2.eval()
        # layer3, layer4 follow the requested mode (train or eval)
        return self

    def forward(self, x):
        with torch.no_grad():
            x = self.stem(x)
            c1 = self.layer1(x)
            c2 = self.layer2(c1)
        # Detach so no gradient flows back into frozen layers
        c3 = self.layer3(c2.detach())
        c4 = self.layer4(c3)
        return c1, c2, c3, c4  # (256,1/4), (512,1/8), (1024,1/16), (2048,1/32)


# PATH B: Frequency-aware edge branch

class EdgeBranch(nn.Module):
    def __init__(self, out_channels=(64, 128, 256, 512)):
        super().__init__()
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32)
        self.register_buffer("sobel_x", sobel_x.view(1, 1, 3, 3))
        self.register_buffer("sobel_y", sobel_y.view(1, 1, 3, 3))

        c1, c2, c3, c4 = out_channels
        self.stem = nn.Sequential(
            nn.Conv2d(5, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, c1, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c1), nn.ReLU(inplace=True),
            nn.Conv2d(c1, c1, 3, padding=1, bias=False),
            nn.BatchNorm2d(c1), nn.ReLU(inplace=True),
        )
        self.block2 = self._block(c1, c2)
        self.block3 = self._block(c2, c3)
        self.block4 = self._block(c3, c4)
        self.edge_head = nn.Conv2d(c1, 1, 1)

    @staticmethod
    def _block(in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )

    def _sobel(self, x):
        gray = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
        sx = F.conv2d(gray, self.sobel_x, padding=1)
        sy = F.conv2d(gray, self.sobel_y, padding=1)
        return torch.cat([sx, sy], dim=1)

    def forward(self, x):
        z = torch.cat([x, self._sobel(x)], dim=1)
        e1 = self.stem(z)
        e2 = self.block2(e1)
        e3 = self.block3(e2)
        e4 = self.block4(e3)
        edge_logit = self.edge_head(e1)
        return (e1, e2, e3, e4), edge_logit


class CrossAttnFusion(nn.Module):
    def __init__(self, dim_a, dim_b, out_dim, num_heads=4):
        super().__init__()
        self.proj_a = nn.Conv2d(dim_a, out_dim, 1)
        self.proj_b = nn.Conv2d(dim_b, out_dim, 1)
        self.norm_a = nn.LayerNorm(out_dim)
        self.norm_b = nn.LayerNorm(out_dim)
        self.attn_a2b = nn.MultiheadAttention(out_dim, num_heads, batch_first=True)
        self.attn_b2a = nn.MultiheadAttention(out_dim, num_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(out_dim * 2, out_dim * 2),
            nn.GELU(),
            nn.Linear(out_dim * 2, out_dim),
        )
        self.norm_out = nn.LayerNorm(out_dim)

    def forward(self, a, b):
        B, _, H, W = a.shape
        a = self.proj_a(a)
        b = self.proj_b(b)
        C = a.size(1)
        a_seq = a.flatten(2).transpose(1, 2)
        b_seq = b.flatten(2).transpose(1, 2)
        a_seq = self.norm_a(a_seq)
        b_seq = self.norm_b(b_seq)
        a_att, _ = self.attn_a2b(a_seq, b_seq, b_seq)
        b_att, _ = self.attn_b2a(b_seq, a_seq, a_seq)
        fused = torch.cat([a_seq + a_att, b_seq + b_att], dim=-1)
        fused = self.ffn(fused) + a_seq
        fused = self.norm_out(fused)
        return fused.transpose(1, 2).reshape(B, C, H, W)


class LightFusion(nn.Module):
    def __init__(self, dim_a, dim_b, out_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(dim_a + dim_b, out_dim, 1, bias=False),
            nn.BatchNorm2d(out_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, a, b):
        return self.proj(torch.cat([a, b], dim=1))



# CRF-as-RNN
class CRFasRNN(nn.Module):
    def __init__(self, num_classes, num_iter=5, ksize=5):
        super().__init__()
        self.num_classes = num_classes
        self.num_iter = num_iter
        self.ksize = ksize

        self.smooth = nn.Conv2d(num_classes, num_classes, ksize,
                                padding=ksize // 2, groups=num_classes, bias=False)
        self.appearance = nn.Conv2d(num_classes + 3, num_classes, ksize,
                                    padding=ksize // 2, bias=False)
        self.w_smooth = nn.Parameter(torch.tensor(1.0))
        self.w_appear = nn.Parameter(torch.tensor(1.0))
        self.compat = nn.Conv2d(num_classes, num_classes, 1, bias=False)

        with torch.no_grad():
            g = self._gaussian(ksize, sigma=1.0)
            for c in range(num_classes):
                self.smooth.weight[c, 0].copy_(g)
            nn.init.xavier_uniform_(self.appearance.weight)
            w = torch.eye(num_classes).neg().view(num_classes, num_classes, 1, 1)
            self.compat.weight.copy_(w)

    @staticmethod
    def _gaussian(ksize, sigma):
        ax = torch.arange(ksize, dtype=torch.float32) - ksize // 2
        yy, xx = torch.meshgrid(ax, ax, indexing="ij")
        g = torch.exp(-(xx * xx + yy * yy) / (2 * sigma * sigma))
        return g / g.sum()

    def forward(self, unary, image):
        if image.shape[-2:] != unary.shape[-2:]:
            image = F.interpolate(image, size=unary.shape[-2:],
                                  mode="bilinear", align_corners=False)
        q = F.softmax(unary, dim=1)
        for _ in range(self.num_iter):
            m_smooth = self.smooth(q)
            m_appear = self.appearance(torch.cat([q, image], dim=1))
            message = self.w_smooth * m_smooth + self.w_appear * m_appear
            pairwise = self.compat(message)
            q = F.softmax(unary - pairwise, dim=1)
        return q


class MultiScaleCRFDecoder(nn.Module):
    def __init__(self, fused_channels, num_classes=NUM_CLASSES,
                 decoder_dim=256, crf_iter=CRF_ITERATIONS):
        super().__init__()
        d = decoder_dim
        f1, f2, f3, f4 = fused_channels

        self.lat4 = nn.Conv2d(f4, d, 1)
        self.lat3 = nn.Conv2d(f3, d, 1)
        self.lat2 = nn.Conv2d(f2, d, 1)
        self.lat1 = nn.Conv2d(f1, d, 1)

        self.smooth4 = self._smooth(d)
        self.smooth3 = self._smooth(d)
        self.smooth2 = self._smooth(d)
        self.smooth1 = self._smooth(d)

        self.head_q = nn.Conv2d(d, num_classes, 1)
        self.up_q2h = nn.Sequential(
            nn.Conv2d(d, d // 2, 3, padding=1, bias=False),
            nn.BatchNorm2d(d // 2), nn.ReLU(inplace=True),
        )
        self.head_h = nn.Conv2d(d // 2, num_classes, 1)
        self.up_h2f = nn.Sequential(
            nn.Conv2d(d // 2, d // 4, 3, padding=1, bias=False),
            nn.BatchNorm2d(d // 4), nn.ReLU(inplace=True),
        )
        self.head_f = nn.Conv2d(d // 4, num_classes, 1)

        self.crf_q = CRFasRNN(num_classes, num_iter=crf_iter)
        self.crf_h = CRFasRNN(num_classes, num_iter=crf_iter)
        self.crf_f = CRFasRNN(num_classes, num_iter=crf_iter)

    @staticmethod
    def _smooth(d):
        return nn.Sequential(
            nn.Conv2d(d, d, 3, padding=1, bias=False),
            nn.BatchNorm2d(d), nn.ReLU(inplace=True),
        )

    def forward(self, fused_feats, image):
        f1, f2, f3, f4 = fused_feats

        p4 = self.smooth4(self.lat4(f4))
        p3 = self.smooth3(self.lat3(f3) + F.interpolate(p4, size=f3.shape[-2:],
                                                        mode="bilinear", align_corners=False))
        p2 = self.smooth2(self.lat2(f2) + F.interpolate(p3, size=f2.shape[-2:],
                                                        mode="bilinear", align_corners=False))
        p1 = self.smooth1(self.lat1(f1) + F.interpolate(p2, size=f1.shape[-2:],
                                                        mode="bilinear", align_corners=False))

        unary_q = self.head_q(p1)
        prob_q = self.crf_q(unary_q, image)

        p_h = F.interpolate(p1, scale_factor=2, mode="bilinear", align_corners=False)
        p_h = self.up_q2h(p_h)
        unary_h = self.head_h(p_h)
        prob_h = self.crf_h(unary_h, image)

        p_f = F.interpolate(p_h, scale_factor=2, mode="bilinear", align_corners=False)
        p_f = self.up_h2f(p_f)
        unary_f = self.head_f(p_f)
        prob_f = self.crf_f(unary_f, image)

        return {
            "unary_q": unary_q, "prob_q": prob_q,
            "unary_h": unary_h, "prob_h": prob_h,
            "unary_f": unary_f, "prob_f": prob_f,
        }


class DualPathCRFNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, decoder_dim=256, crf_iter=CRF_ITERATIONS):
        super().__init__()
        self.backbone = ResNet50Backbone()
        self.edge = EdgeBranch(out_channels=(64, 128, 256, 512))

        self.fuse1 = LightFusion(256, 64, 256)
        self.fuse2 = LightFusion(512, 128, 256)
        self.fuse3 = CrossAttnFusion(1024, 256, 256, num_heads=4)
        self.fuse4 = CrossAttnFusion(2048, 512, 256, num_heads=4)

        self.decoder = MultiScaleCRFDecoder(
            fused_channels=(256, 256, 256, 256),
            num_classes=num_classes,
            decoder_dim=decoder_dim,
            crf_iter=crf_iter,
        )

    def forward(self, image):
        c1, c2, c3, c4 = self.backbone(image)
        (e1, e2, e3, e4), edge_logit = self.edge(image)
        f1 = self.fuse1(c1, e1)
        f2 = self.fuse2(c2, e2)
        f3 = self.fuse3(c3, e3)
        f4 = self.fuse4(c4, e4)
        out = self.decoder((f1, f2, f3, f4), image)
        out["edge_logit"] = edge_logit
        return out



# Loss functions

def seg_loss_from_logits(logits, target):
    logits_up = F.interpolate(logits, size=target.shape[-2:],
                              mode="bilinear", align_corners=False)
    return F.cross_entropy(logits_up, target, ignore_index=IGNORE_INDEX)


def seg_loss_from_probs(probs, target, eps=1e-8):
    probs_up = F.interpolate(probs, size=target.shape[-2:],
                             mode="bilinear", align_corners=False)
    log_probs = torch.log(probs_up.clamp_min(eps))
    return F.nll_loss(log_probs, target, ignore_index=IGNORE_INDEX)


def compute_total_loss(out, target, boundary):
    loss_main = seg_loss_from_probs(out["prob_f"], target)
    loss_aux = seg_loss_from_probs(out["prob_q"], target) + \
               seg_loss_from_probs(out["prob_h"], target)
    loss_unary = seg_loss_from_logits(out["unary_f"], target)

    edge_logit = out["edge_logit"]
    edge_up = F.interpolate(edge_logit, size=boundary.shape[-2:],
                            mode="bilinear", align_corners=False).squeeze(1)
    pos_weight = torch.tensor(5.0, device=edge_up.device)
    loss_edge = F.binary_cross_entropy_with_logits(edge_up, boundary, pos_weight=pos_weight)

    total = loss_main + AUX_LOSS_WEIGHT * (loss_aux + loss_unary) + EDGE_LOSS_WEIGHT * loss_edge
    return total, {
        "loss": total.item(),
        "main": loss_main.item(),
        "aux": loss_aux.item(),
        "unary": loss_unary.item(),
        "edge": loss_edge.item(),
    }


# Metrics

class ConfusionMatrix:
    def __init__(self, num_classes):
        self.n = num_classes
        self.mat = torch.zeros(num_classes * num_classes, dtype=torch.long)

    def update(self, pred, target):
        pred = pred.flatten()
        target = target.flatten()
        valid = (target >= 0) & (target < self.n)
        inds = self.n * target[valid].long() + pred[valid].long()
        self.mat += torch.bincount(inds.cpu(), minlength=self.n * self.n)

    def compute(self):
        mat = self.mat.reshape(self.n, self.n).float()
        tp = mat.diag()
        fp = mat.sum(0) - tp
        fn = mat.sum(1) - tp
        iou = tp / (tp + fp + fn + 1e-10)
        dice = 2 * tp / (2 * tp + fp + fn + 1e-10)
        pix_acc = tp.sum() / (mat.sum() + 1e-10)
        return {
            "mIoU": iou.mean().item(),
            "mDice": dice.mean().item(),
            "pixel_acc": pix_acc.item(),
            "per_class_iou": {CLASS_NAMES[i]: iou[i].item() for i in range(self.n)},
        }

    def reset(self):
        self.mat.zero_()

def build_optimizer(model, lr, wd):
    """Two param groups: backbone (unfrozen) at lr * BACKBONE_LR_MULT, rest at lr."""
    backbone_params = []
    other_params = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "backbone" in name:
            backbone_params.append(p)
        else:
            other_params.append(p)

    param_groups = [
        {"params": backbone_params, "lr": lr * BACKBONE_LR_MULT, "group_name": "backbone"},
        {"params": other_params,    "lr": lr,                     "group_name": "head"},
    ]
    print(f"Optimizer: backbone params={len(backbone_params)} (lr={lr * BACKBONE_LR_MULT:.1e}), "
          f"head params={len(other_params)} (lr={lr:.1e})")
    return torch.optim.AdamW(param_groups, weight_decay=wd)


def lr_at(step, total_steps, warmup_steps, base_lr):
    if step < warmup_steps:
        return base_lr * (step + 1) / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * base_lr * (1 + math.cos(math.pi * progress))



# Train / Val loops

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train_one_epoch(model, loader, optimizer, scaler, device, epoch,
                    total_steps, warmup_steps, step_offset):
    model.train()
    running = {"loss": 0, "main": 0, "aux": 0, "unary": 0, "edge": 0}
    pbar = tqdm(loader, desc=f"train ep{epoch}", leave=True, dynamic_ncols=True)

    for i, (img, tgt, boundary) in enumerate(pbar):
        step = step_offset + i

        # Update LR per param group with correct base
        for g in optimizer.param_groups:
            if g.get("group_name") == "backbone":
                g["lr"] = lr_at(step, total_steps, warmup_steps, BASE_LR * BACKBONE_LR_MULT)
            else:
                g["lr"] = lr_at(step, total_steps, warmup_steps, BASE_LR)

        img = img.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)
        boundary = boundary.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=MIXED_PRECISION):
            out = model(img)
            loss, logs = compute_total_loss(out, tgt, boundary)

        if MIXED_PRECISION:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()

        for k in running:
            running[k] += logs[k]

        n = i + 1
        pbar.set_postfix({
            "loss":  f"{running['loss']  / n:.3f}",
            "main":  f"{running['main']  / n:.3f}",
            "edge":  f"{running['edge']  / n:.3f}",
            "lr_h":  f"{optimizer.param_groups[1]['lr']:.1e}",
            "lr_bb": f"{optimizer.param_groups[0]['lr']:.1e}",
        })

    n = len(loader)
    return {k: v / n for k, v in running.items()}


@torch.no_grad()
def evaluate(model, loader, device, report_before_crf=True, epoch=None):
    model.eval()
    cm_after = ConfusionMatrix(NUM_CLASSES)
    cm_before = ConfusionMatrix(NUM_CLASSES) if report_before_crf else None

    desc = f"val ep{epoch}" if epoch is not None else "val"
    for img, tgt, _ in tqdm(loader, desc=desc, leave=True, dynamic_ncols=True):
        img = img.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=MIXED_PRECISION):
            out = model(img)

        probs_after = F.interpolate(out["prob_f"], size=tgt.shape[-2:],
                                    mode="bilinear", align_corners=False)
        pred_after = probs_after.argmax(1)
        cm_after.update(pred_after, tgt)

        if report_before_crf:
            logits_before = F.interpolate(out["unary_f"], size=tgt.shape[-2:],
                                          mode="bilinear", align_corners=False)
            pred_before = logits_before.argmax(1)
            cm_before.update(pred_before, tgt)

    metrics = {"after_crf": cm_after.compute()}
    if report_before_crf:
        metrics["before_crf"] = cm_before.compute()
    return metrics


def load_existing_log(log_path: Path):
    """Load all existing log entries from JSONL. Returns list of dicts."""
    entries = []
    if log_path.is_file():
        with open(log_path) as f:
            for line in f:
                line = line.strip()
                if line:
                    entries.append(json.loads(line))
    return entries


def append_log(log_path: Path, entry: dict):
    """Append a single JSON entry to the log file."""
    with open(log_path, "a") as f:
        f.write(json.dumps(entry) + "\n")


def generate_final_outputs(log_path: Path, output_dir: Path):
    """Generate training plots and summary JSON from the accumulated log."""
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib not available, skipping plot generation.")
        return

    entries = load_existing_log(log_path)
    if not entries:
        print("No log entries found, skipping final outputs.")
        return

    epochs = [e["epoch"] for e in entries]
    train_loss = [e["train"]["loss"] for e in entries]
    train_main = [e["train"]["main"] for e in entries]
    train_edge = [e["train"]["edge"] for e in entries]

    val_epochs, miou_after, miou_before = [], [], []
    mdice_after, pix_acc_after = [], []
    per_class_after = {}

    for e in entries:
        if "val" in e:
            val_epochs.append(e["epoch"])
            miou_after.append(e["val"]["after_crf"]["mIoU"])
            mdice_after.append(e["val"]["after_crf"]["mDice"])
            pix_acc_after.append(e["val"]["after_crf"]["pixel_acc"])
            if "before_crf" in e["val"]:
                miou_before.append(e["val"]["before_crf"]["mIoU"])
            for cls_name, iou_val in e["val"]["after_crf"]["per_class_iou"].items():
                per_class_after.setdefault(cls_name, []).append(iou_val)

    fig_dir = output_dir / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)

    #   Plot 1: Training losses  
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(epochs, train_loss, label="total loss", linewidth=2)
    ax.plot(epochs, train_main, label="main (CE)", linewidth=1.5, alpha=0.8)
    ax.plot(epochs, train_edge, label="edge (BCE)", linewidth=1.5, alpha=0.8)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training Losses")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(fig_dir / "train_losses.png", dpi=150)
    plt.close(fig)

    if val_epochs:
        #   Plot 2: mIoU (before vs after CRF)  
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(val_epochs, miou_after, label="after CRF", linewidth=2, marker="o", markersize=4)
        if miou_before:
            ax.plot(val_epochs, miou_before, label="before CRF", linewidth=2,
                    marker="s", markersize=4, alpha=0.7)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("mIoU")
        ax.set_title("Validation mIoU: Before vs After CRF")
        ax.legend()
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        fig.savefig(fig_dir / "val_miou.png", dpi=150)
        plt.close(fig)

        #  3: mDice and Pixel Accuracy -- 
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        ax1.plot(val_epochs, mdice_after, linewidth=2, marker="o", markersize=4, color="tab:green")
        ax1.set_xlabel("Epoch")
        ax1.set_ylabel("mDice")
        ax1.set_title("Validation mDice (after CRF)")
        ax1.grid(True, alpha=0.3)

        ax2.plot(val_epochs, pix_acc_after, linewidth=2, marker="o", markersize=4, color="tab:orange")
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("Pixel Accuracy")
        ax2.set_title("Validation Pixel Accuracy (after CRF)")
        ax2.grid(True, alpha=0.3)
        fig.tight_layout()
        fig.savefig(fig_dir / "val_dice_pixacc.png", dpi=150)
        plt.close(fig)

        #  t 4: Per-class IoU bar chart (final epoch)    
        if per_class_after:
            cls_names = list(per_class_after.keys())
            final_ious = [per_class_after[c][-1] for c in cls_names]
            sorted_pairs = sorted(zip(cls_names, final_ious), key=lambda x: x[1], reverse=True)
            cls_names_sorted, ious_sorted = zip(*sorted_pairs)

            fig, ax = plt.subplots(figsize=(12, 6))
            colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(cls_names_sorted)))
            bars = ax.barh(range(len(cls_names_sorted)), ious_sorted, color=colors)
            ax.set_yticks(range(len(cls_names_sorted)))
            ax.set_yticklabels(cls_names_sorted)
            ax.set_xlabel("IoU")
            ax.set_title(f"Per-Class IoU (after CRF, epoch {val_epochs[-1]})")
            ax.set_xlim(0, 1)
            for bar, val in zip(bars, ious_sorted):
                ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                        f"{val:.3f}", va="center", fontsize=9)
            ax.grid(True, axis="x", alpha=0.3)
            fig.tight_layout()
            fig.savefig(fig_dir / "per_class_iou.png", dpi=150)
            plt.close(fig)

    #  
    summary = {
        "total_epochs_trained": len(entries),
        "final_epoch": entries[-1]["epoch"],
        "final_train_loss": entries[-1]["train"]["loss"],
    }
    if val_epochs:
        best_idx = int(np.argmax(miou_after))
        summary["best_val_epoch"] = val_epochs[best_idx]
        summary["best_val_mIoU_after_crf"] = miou_after[best_idx]
        if miou_before:
            summary["best_val_mIoU_before_crf"] = miou_before[best_idx]
        summary["final_val_mIoU_after_crf"] = miou_after[-1]
        summary["final_val_mDice_after_crf"] = mdice_after[-1]
        summary["final_val_pixel_acc"] = pix_acc_after[-1]
        if per_class_after:
            summary["final_per_class_iou"] = {c: per_class_after[c][-1] for c in per_class_after}

    with open(output_dir / "training_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print(f"Figures saved to: {fig_dir}")
    print(f"Summary saved to: {output_dir / 'training_summary.json'}")


# =============================================================================
# Main
# =============================================================================
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-root", default=DATA_ROOT)
    parser.add_argument("--output-dir", default=OUTPUT_DIR)
    parser.add_argument("--epochs", type=int, default=EPOCHS)
    parser.add_argument("--batch-size", type=int, default=BATCH_SIZE)
    parser.add_argument("--lr", type=float, default=BASE_LR)
    parser.add_argument("--resume", default="", help="explicit path to checkpoint")
    parser.add_argument("--no-auto-resume", action="store_true",
                        help="disable auto-resume even if AUTO_RESUME=True")
    args, _ = parser.parse_known_args()

    set_seed(SEED)
    os.makedirs(args.output_dir, exist_ok=True)
    log_path = Path(args.output_dir) / "train_log.jsonl"
    ckpt_last = Path(args.output_dir) / "last.pt"
    ckpt_best = Path(args.output_dir) / "best.pt"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    train_ds = CityscapesFlat(args.data_root, split="train", is_train=True)
    val_ds = CityscapesFlat(args.data_root, split="val", is_train=False)
    print(f"Train: {len(train_ds)} images | Val: {len(val_ds)} images")

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                              persistent_workers=NUM_WORKERS > 0)
    val_loader = DataLoader(val_ds, batch_size=max(1, args.batch_size // 2),
                            shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
                            persistent_workers=NUM_WORKERS > 0)

    model = DualPathCRFNet().to(device)
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"Params: trainable={n_trainable/1e6:.2f}M total={n_total/1e6:.2f}M")

    optimizer = build_optimizer(model, args.lr, WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=MIXED_PRECISION)

    #    --   -- 
    # Resume logic: explicit --resume flag > AUTO_RESUME from last.pt
    #       
    start_epoch = 0
    best_miou = 0.0
    no_improve_count = 0  # for early stopping

    resume_path = None
    if args.resume and Path(args.resume).is_file():
        resume_path = Path(args.resume)
    elif AUTO_RESUME and not args.no_auto_resume and ckpt_last.is_file():
        resume_path = ckpt_last
        print(f"[AUTO_RESUME] Found {ckpt_last}, resuming automatically.")

    if resume_path is not None:
        ck = torch.load(resume_path, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"])
        if "optim" in ck:
            optimizer.load_state_dict(ck["optim"])
        if "scaler" in ck:
            scaler.load_state_dict(ck["scaler"])
        start_epoch = ck.get("epoch", 0) + 1
        best_miou = ck.get("best_miou", 0.0)
        no_improve_count = ck.get("no_improve_count", 0)
        print(f"Resumed from epoch {start_epoch}, best mIoU={best_miou:.4f}, "
              f"no_improve={no_improve_count}/{EARLY_STOP_PATIENCE}")

    steps_per_epoch = len(train_loader)
    total_steps = steps_per_epoch * args.epochs
    warmup_steps = steps_per_epoch * WARMUP_EPOCHS

    #    --      
    # Training loop
    #  --  --   --   
    for epoch in range(start_epoch, args.epochs):
        t0 = time.time()
        print(f"\n{'='*60}")
        print(f"  Epoch {epoch+1}/{args.epochs}  |  best mIoU={best_miou:.4f}  |  "
              f"patience={no_improve_count}/{EARLY_STOP_PATIENCE}")
        print(f"{'='*60}")

        step_offset = epoch * steps_per_epoch
        train_stats = train_one_epoch(model, train_loader, optimizer, scaler,
                                      device, epoch + 1, total_steps, warmup_steps, step_offset)
        elapsed = time.time() - t0
        print(f"  train: " + " ".join(f"{k}={v:.4f}" for k, v in train_stats.items())
              + f"  ({elapsed/60:.1f} min)")

        log_entry = {
            "epoch": epoch + 1,
            "train": train_stats,
            "elapsed_min": round(elapsed / 60, 2),
            "lr_head": optimizer.param_groups[1]["lr"],
            "lr_backbone": optimizer.param_groups[0]["lr"],
        }

        # Validation
        if (epoch + 1) % VAL_EVERY == 0:
            val_metrics = evaluate(model, val_loader, device,
                                   report_before_crf=True, epoch=epoch + 1)
            after = val_metrics["after_crf"]
            before = val_metrics["before_crf"]
            print(f"  val (after  CRF): mIoU={after['mIoU']:.4f}  mDice={after['mDice']:.4f}  "
                  f"pixAcc={after['pixel_acc']:.4f}")
            print(f"  val (before CRF): mIoU={before['mIoU']:.4f}  mDice={before['mDice']:.4f}  "
                  f"pixAcc={before['pixel_acc']:.4f}")
            log_entry["val"] = val_metrics

            if after["mIoU"] > best_miou:
                best_miou = after["mIoU"]
                no_improve_count = 0
                torch.save({
                    "model": model.state_dict(),
                    "optim": optimizer.state_dict(),
                    "scaler": scaler.state_dict(),
                    "epoch": epoch,
                    "best_miou": best_miou,
                    "no_improve_count": no_improve_count,
                    "val_metrics": val_metrics,
                }, ckpt_best)
                print(f"  ** New best mIoU={best_miou:.4f} -> saved {ckpt_best.name}")
            else:
                no_improve_count += 1
                print(f"  No improvement ({no_improve_count}/{EARLY_STOP_PATIENCE})")

        # Always save last.pt (for resume)
        torch.save({
            "model": model.state_dict(),
            "optim": optimizer.state_dict(),
            "scaler": scaler.state_dict(),
            "epoch": epoch,
            "best_miou": best_miou,
            "no_improve_count": no_improve_count,
        }, ckpt_last)

        # Append to log (safe across pause/resume)
        append_log(log_path, log_entry)

        # Early stopping check
        if no_improve_count >= EARLY_STOP_PATIENCE:
            print(f"\n*** Early stopping triggered at epoch {epoch+1} "
                  f"(no improvement for {EARLY_STOP_PATIENCE} val epochs) ***")
            break

    #        
    # Training complete -> generate figures and summary
    #  --    --  
    print(f"\n{'='*60}")
    print(f"Training finished. Best val mIoU (after CRF) = {best_miou:.4f}")
    print(f"Checkpoints: {ckpt_best}  |  {ckpt_last}")
    print(f"{'='*60}")

    generate_final_outputs(log_path, Path(args.output_dir))


if __name__ == "__main__":
    main()